In [21]:
# 导入 pandas 库用于数据处理
import pandas as pd

In [22]:
# 读取 Netflix 数据集 CSV 文件
df = pd.read_csv("netflix_titles.csv")
# 查看所有列名
df.columns

Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description'],
      dtype='str')

### 问题1
#### .info()使用`, .describe()` 和 `检查 DataFrame`.head()

In [23]:
# 查看数据集前5行，初步了解数据结构和内容
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [24]:
# 查看数值型列的统计摘要（计数、均值、标准差、最小值、四分位数、最大值）
df.describe()

,release_year
count,8807.000000
mean,2014.180198
std,8.819312
min,1925.000000
25%,2013.000000
50%,2017.000000
75%,2019.000000
max,2021.000000


In [25]:
# 查看数据集整体信息：行数、列数、每列的非空值数量和数据类型
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 825.8 KB


### 问题2
#### 逐列识别并处理缺失值

In [26]:
# 统计每列的缺失值数量
df.isnull().sum()

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

In [27]:
# 按缺失值数量从高到低排序，便于直观查看哪些列缺失最严重
missing_values = df.isnull().sum().sort_values(ascending=False)
missing_values

director        2634
country          831
cast             825
date_added        10
rating             4
duration           3
show_id            0
type               0
title              0
release_year       0
listed_in          0
description        0
dtype: int64

In [28]:
# 计算每列缺失值的百分比，更直观地了解缺失比例
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_percent.sort_values(ascending=False)

director        29.908028
country          9.435676
cast             9.367549
date_added       0.113546
rating           0.045418
duration         0.034064
show_id          0.000000
type             0.000000
title            0.000000
release_year     0.000000
listed_in        0.000000
description      0.000000
dtype: float64

In [29]:
# 将 director（导演）列的缺失值填充为 "Unknown"
df["director"] = df["director"].fillna("Unknown")

In [30]:
# 将 cast（演员）列的缺失值填充为 "Not Available"
df["cast"] = df["cast"].fillna("Not Available")

In [31]:
# 将 country（国家）列的缺失值填充为 "Unknown"
df["country"] = df["country"].fillna("Unknown")

In [32]:
# 将 date_added（添加日期）列的缺失值用前一行值进行前向填充
df["date_added"] = df["date_added"].ffill()

In [33]:
# 将 rating（评级）列的缺失值用众数（出现频率最高的值）填充
df["rating"] = df["rating"].fillna(df["rating"].mode()[0])

In [34]:
# 删除 duration（时长）列中含有缺失值的行（仅3行，影响很小）
df = df.dropna(subset=["duration"])

### 问题3
#### 修复混合类型列（例如，持续时间存储为"90 min"）

In [35]:
# 将 duration 列按空格拆分为数值和单位两列（如 "90 min" → "90" 和 "min"）
df[["duration_value", "duration_type"]] = df["duration"].str.split(" ", expand=True)

In [36]:
# 将 duration_value 列转换为数值类型，无法转换的设为 NaN
df["duration_value"] = pd.to_numeric(df["duration_value"], errors="coerce")

In [37]:
# 统一 duration_type 的格式：min→Minutes，Season/Seasons→Seasons
df["duration_type"] = df["duration_type"].replace({
    "min": "Minutes",
    "Season": "Seasons",
    "Seasons": "Seasons"
})

### 问题4
#### 将日期列解析为正确的datetime对象并保存

In [38]:
# 将 date_added 列转换为 datetime 类型，无法解析的设为 NaT
df["date_added"] = pd.to_datetime(df["date_added"], errors="coerce")

In [39]:
# 将清洗后的数据保存为新的 CSV 文件
df.to_csv('cleaned-data.csv')